# 🚀 Approach 1: Regularized Linear Classifiers (10-Fold Repeated CV + Outlier Clipping)

## 🧠 The Core Hypothesis:
On high-dimensional tabular datasets (344+ anonymized features) with extreme class imbalance (96.2% Negative vs 3.8% Positive):
1. **Global Hyperplane vs Fragmented Trees:** Individual features carry weak linear signals (e.g. $+0.01$ or $-0.01$). Linear models combine all 344 weak signals globally ($z = \sum w_i x_i + b$), whereas trees with `depth=5` can only inspect 5 features per path.
2. **10-Fold Stratified Cross-Validation (90% Train per Fold):** Each model sees 90% of the positive minority samples (~2,707 positives vs 2,406 in 5-fold), providing richer training signal.
3. **Multi-Seed Averaging (3 Seeds x 10 Folds = 30 Models per Family):** Reduces prediction variance dramatically across unseen test data.
4. **1% - 99% Outlier Clipping:** Prevents extreme outliers from skewing linear weights $w_i$.
5. **High-Resolution Threshold Scan (step=0.001):** Pin-points the exact mathematical maximum F1 score.

**Hardware:** 100% CPU Friendly (Runs in ~2-3 minutes).


In [ ]:
import os, sys, time, gc, warnings
import numpy as np
import pandas as pd
from scipy.stats import skew, kurtosis
warnings.filterwarnings('ignore')

# ML Models & Evaluation
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression, SGDClassifier, RidgeClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score

SEEDS = [42, 123, 456]
N_FOLDS = 10
print('=' * 75)
print('  APPROACH 1: 10-FOLD MULTI-SEED REGULARIZED LINEAR ENSEMBLE')
print('=' * 75)


## 1. Load Data & Clean Features

In [ ]:
CANDIDATE_DIRS = [
    '/kaggle/input/competitions/pstu-data-thon-2026-vol-1',
    '/kaggle/input/pstu-data-thon-2026-vol-1',
    'pstu-data-thon-2026-vol-1',
    '../input/competitions/pstu-data-thon-2026-vol-1',
    './Dataset',
    '.'
]
DATA_DIR = next((d for d in CANDIDATE_DIRS if os.path.exists(os.path.join(d, 'train.csv'))), None)
if DATA_DIR is None: raise FileNotFoundError('train.csv not found')

train_raw = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test_raw  = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))
print(f'Train Shape: {train_raw.shape} | Test Shape: {test_raw.shape}')

TARGET_COL = 'TARGET'
y = train_raw[TARGET_COL].copy()
if 'id' in test_raw.columns:
    test_ids = test_raw['id'].copy()
    X_test_raw = test_raw.drop(columns=['id'])
else:
    test_ids = pd.Series(range(len(test_raw)), name='id')
    X_test_raw = test_raw.copy()
X_train_raw = train_raw.drop(columns=[TARGET_COL])

feat_cols = [c for c in X_train_raw.columns if c.startswith('feat_')]
cat_cols  = X_train_raw[feat_cols].select_dtypes(include=['object']).columns.tolist()
num_cols  = [c for c in feat_cols if c not in cat_cols]

# Clean zero-variance & hash-duplicates
X_num_tr = X_train_raw[num_cols].apply(pd.to_numeric, errors='coerce').astype(np.float32)
X_num_te = X_test_raw[num_cols].apply(pd.to_numeric, errors='coerce').astype(np.float32)

variances = X_num_tr.var()
zero_var = variances[variances <= 1e-12].index.tolist()
arr_tr = X_num_tr.values.astype(np.float64)
dup_drop = set()
sigs = {}
for i, c in enumerate(num_cols):
    if c in zero_var: continue
    col = arr_tr[:, i]
    sig = (hash(col[:500].tobytes()), hash(col[500:1000].tobytes()), int(col.var()*1e6))
    if sig in sigs:
        j = sigs[sig]
        if np.array_equal(col, arr_tr[:, j]): dup_drop.add(c)
    else: sigs[sig] = i
all_drop = set(zero_var) | dup_drop
keep_num = [c for c in num_cols if c not in all_drop]
X_num_tr = X_num_tr[keep_num].fillna(0)
X_num_te = X_num_te[keep_num].fillna(0)
print(f'Kept Numerical Features: {len(keep_num)}')


## 2. Outlier Quantile Clipping & Robust Feature Preprocessing

In [ ]:
# Outlier Clipping (1% to 99% Percentiles)
p_low = np.percentile(X_num_tr, 1, axis=0)
p_high = np.percentile(X_num_tr, 99, axis=0)
X_num_tr_clipped = np.clip(X_num_tr.values, p_low, p_high)
X_num_te_clipped = np.clip(X_num_te.values, p_low, p_high)

X_num_tr = pd.DataFrame(X_num_tr_clipped, columns=keep_num)
X_num_te = pd.DataFrame(X_num_te_clipped, columns=keep_num)

# Row Statistics (Global Density)
def compute_row_stats(df_num):
    arr = df_num.values.astype(np.float64)
    stats = pd.DataFrame(index=df_num.index)
    stats['row_mean'] = arr.mean(axis=1).astype(np.float32)
    stats['row_std']  = arr.std(axis=1).astype(np.float32)
    stats['row_iqr']  = (np.percentile(arr, 75, axis=1) - np.percentile(arr, 25, axis=1)).astype(np.float32)
    stats['row_zero'] = (arr == 0).sum(axis=1).astype(np.float32)
    return stats

df_row_tr = compute_row_stats(X_num_tr)
df_row_te = compute_row_stats(X_num_te)

# Categorical Frequency Encoding
df_cat_tr = pd.DataFrame(index=X_train_raw.index)
df_cat_te = pd.DataFrame(index=X_test_raw.index)
for col in cat_cols:
    freq_map = pd.concat([X_train_raw[col], X_test_raw[col]]).value_counts(normalize=True).to_dict()
    df_cat_tr[f"{col}_freq"] = X_train_raw[col].map(freq_map).fillna(0).astype(np.float32)
    df_cat_te[f"{col}_freq"] = X_test_raw[col].map(freq_map).fillna(0).astype(np.float32)

X_tr_combined = pd.concat([X_num_tr, df_cat_tr, df_row_tr], axis=1).fillna(0)
X_te_combined = pd.concat([X_num_te, df_cat_te, df_row_te], axis=1).fillna(0)

# RobustScaler (Quantile 5% to 95% scaling)
scaler = RobustScaler(quantile_range=(5.0, 95.0))
X_tr_scaled = np.nan_to_num(scaler.fit_transform(X_tr_combined), nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
X_te_scaled = np.nan_to_num(scaler.transform(X_te_combined), nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
print(f'Final Scaled Matrix: Train {X_tr_scaled.shape} | Test {X_te_scaled.shape}')


## 3. Train 10-Fold Multi-Seed Linear Classifiers (30 Models per Family)

In [ ]:
# 4 Diverse Regularized Linear Model Families
models_dict = {
    'Logistic_L2': lambda s: LogisticRegression(C=0.05, penalty='l2', solver='lbfgs', class_weight='balanced', max_iter=1000, random_state=s),
    'Logistic_L1_Lasso': lambda s: LogisticRegression(C=0.02, penalty='l1', solver='saga', class_weight='balanced', max_iter=500, random_state=s, n_jobs=-1),
    'SGD_ElasticNet': lambda s: SGDClassifier(loss='log_loss', penalty='elasticnet', alpha=1e-3, l1_ratio=0.20, class_weight='balanced', max_iter=1000, random_state=s),
    'Ridge_Calibrated': lambda s: CalibratedClassifierCV(RidgeClassifier(alpha=20.0, class_weight='balanced', random_state=s), method='sigmoid', cv=3)
}

oof_all_models = {m: np.zeros(len(y), dtype=np.float32) for m in models_dict}
test_all_models = {m: np.zeros(len(X_te_scaled), dtype=np.float32) for m in models_dict}

t_start_total = time.time()

for m_name, model_fn in models_dict.items():
    t_m = time.time()
    print(f'\nTraining {m_name} across {len(SEEDS)} Seeds x {N_FOLDS} Folds ({len(SEEDS)*N_FOLDS} total models)...')
    
    oof_seeds = np.zeros(len(y), dtype=np.float32)
    test_seeds = np.zeros(len(X_te_scaled), dtype=np.float32)
    
    for seed_idx, seed in enumerate(SEEDS):
        skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
        
        for fold, (tr_idx, va_idx) in enumerate(skf.split(X_tr_scaled, y)):
            X_tr, y_tr = X_tr_scaled[tr_idx], y.iloc[tr_idx].values
            X_va, y_va = X_tr_scaled[va_idx], y.iloc[va_idx].values
            
            clf = model_fn(seed + fold)
            clf.fit(X_tr, y_tr)
            
            va_p = clf.predict_proba(X_va)[:, 1]
            te_p = clf.predict_proba(X_te_scaled)[:, 1]
            
            oof_seeds[va_idx] += va_p / len(SEEDS)
            test_seeds += te_p / (len(SEEDS) * N_FOLDS)
            
    oof_all_models[m_name] = oof_seeds
    test_all_models[m_name] = test_seeds
    auc = roc_auc_score(y, oof_seeds)
    print(f'  {m_name} 10-Fold OOF AUC: {auc:.5f} [{time.time() - t_m:.1f}s]')

print(f'\nAll 120 model training passes finished in {time.time() - t_start_total:.1f}s!')


## 4. Multi-Family Linear Ensemble & High-Resolution Threshold Optimization

In [ ]:
# Soft-Voting Ensemble
oof_linear_ens = np.mean([oof_all_models[m] for m in models_dict], axis=0)
test_linear_ens = np.mean([test_all_models[m] for m in models_dict], axis=0)

# High-Resolution Threshold Scan (step=0.001)
thresholds = np.arange(0.05, 0.95, 0.001)
best_f1, best_t = 0.0, 0.5

for t in thresholds:
    b = (oof_linear_ens >= t).astype(int)
    if b.sum() == 0: continue
    f = f1_score(y.values, b)
    if f > best_f1:
        best_f1, best_t = f, t

print('=' * 75)
print(f'  10-FOLD LINEAR ENSEMBLE OOF ROC-AUC: {roc_auc_score(y, oof_linear_ens):.5f}')
print(f'  🏆 OPTIMAL F1 THRESHOLD:             t = {best_t:.4f}')
print(f'  🏆 PEAK OOF F1-SCORE:                F1 = {best_f1:.5f}')
print('=' * 75)


## 5. Export Submission

In [ ]:
OUT_DIR = '/kaggle/working' if os.path.isdir('/kaggle/working') else '.'
os.makedirs(OUT_DIR, exist_ok=True)

binary_preds = (test_linear_ens >= best_t).astype(int)
sub = pd.DataFrame({'id': test_ids.values, 'TARGET': binary_preds})
sub.to_csv(os.path.join(OUT_DIR, 'submission.csv'), index=False)

sub_prob = pd.DataFrame({'id': test_ids.values, 'TARGET': test_linear_ens})
sub_prob.to_csv(os.path.join(OUT_DIR, 'submission_prob.csv'), index=False)

print(f'Saved submission.csv with {int(binary_preds.sum()):,} predicted positives ({binary_preds.mean():.2%}).')
print(sub.head(10))
